# Milestone 3 — Hyperparameter sweep and detection gallery
**IE 7615 · Group 1 · Dario Garza**

This notebook completes Dario's two assigned tasks:

1. Compare **confidence** and **NMS IoU** thresholds using the team's fixed YOLOv8 checkpoint.
2. Produce **at least ten annotated multi-face test examples**, with confidence scores and explanatory captions.

The sweep uses validation to select settings. Only the baseline and selected settings are evaluated on test.
Class-aware and class-agnostic NMS are compared as an extra control for overlapping identity predictions.

**Required:** the updated project ZIP and Aditi's trained `best.pt`, or `milestone03_training_results.zip`.
The project ZIP has the data but does not include those trained YOLO weights. Milestone 1 classification weights cannot replace them.

Run cells in order. In Colab, choose a GPU under **Runtime → Change runtime type** if one is available; CPU also works.


## 1. Locate the project

If you opened this notebook directly in Colab, this cell asks you to upload the updated project ZIP.
If the project is already extracted locally or in Colab, it is reused. No dataset regeneration is needed.


In [ ]:
import importlib.util
import json
import os
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = (importlib.util.find_spec("google") is not None and
            importlib.util.find_spec("google.colab") is not None)

def find_project():
    candidates = [Path.cwd(), Path.cwd().parent]
    for parent in [Path.cwd(), Path("/content")]:
        if parent.is_dir():
            candidates.extend(path for path in parent.iterdir() if path.is_dir())
    for candidate in candidates:
        if (candidate / "project_milestone03/hyperparameter_sweep.py").is_file() and (candidate / "project_milestone02/dataset/data.yaml").is_file():
            return candidate.resolve()
    return None

project_dir = find_project()
if project_dir is None:
    if not IN_COLAB:
        raise FileNotFoundError("Extract the updated project ZIP and open this notebook from its project_milestone03 folder.")
    from google.colab import files
    print("Upload the updated IE7615 project ZIP containing hyperparameter_sweep.py and the dataset.")
    uploaded_project = files.upload()
    project_archives = [Path(name) for name in uploaded_project if name.lower().endswith(".zip")]
    if len(project_archives) != 1:
        raise ValueError("Upload exactly one updated project ZIP.")
    destination = Path.cwd().resolve()
    with zipfile.ZipFile(project_archives[0]) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError("The ZIP contains an unsafe extraction path.")
        archive.extractall(destination)
    project_dir = find_project()
    if project_dir is None:
        raise FileNotFoundError("The ZIP does not contain the updated hyperparameter script and Milestone 2 dataset.")

print("Project:", project_dir)


## 2. Install the sweep dependencies

Use the same Ultralytics release as the team's existing baseline evaluation. Keep Colab's installed PyTorch/CUDA environment.


In [ ]:
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(project_dir / "project_milestone03/requirements_hyperparameters.txt"),
], check=True)

import pandas as pd
import torch
import ultralytics
from IPython.display import Image, Markdown, display

sys.path.insert(0, str(project_dir / "project_milestone03"))
from hyperparameter_sweep import audit_dataset, resolve_checkpoint, run_experiment, sha256

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 3. Supply Aditi's checkpoint

Choose **one** input: `best.pt` or the training-results ZIP containing `best.pt`.
This cell automatically requests an upload in Colab if the file is not already present.

For local execution, set `checkpoint_input` below to your file path.


In [ ]:
checkpoint_input = Path(os.environ.get(
    "M3_CHECKPOINT_INPUT", str(project_dir / "project_milestone03/weights/best.pt")
)).expanduser()

if not checkpoint_input.is_file():
    if not IN_COLAB:
        raise FileNotFoundError("Set checkpoint_input to Aditi's best.pt or milestone03_training_results.zip.")
    from google.colab import files
    print("Upload Aditi's best.pt OR milestone03_training_results.zip.")
    uploaded_checkpoint = files.upload()
    checkpoint_candidates = [Path(name) for name in uploaded_checkpoint if Path(name).suffix.lower() in (".pt", ".zip")]
    if len(checkpoint_candidates) != 1:
        raise ValueError("Upload exactly one .pt checkpoint or training-results ZIP.")
    checkpoint_input = checkpoint_candidates[0].resolve()

checkpoint_path = checkpoint_input if checkpoint_input.suffix.lower() == ".pt" else None
training_zip_path = checkpoint_input if checkpoint_input.suffix.lower() == ".zip" else None
if checkpoint_path is None and training_zip_path is None:
    raise ValueError("Checkpoint input must be a .pt file or a .zip file.")
print("Checkpoint input:", checkpoint_input.name)


## 4. Verify the dataset and define the experiment

The default grid contains **4 confidence values × 3 NMS IoU values × 2 class modes = 24 configurations**.
The image size remains 640, matching the baseline. No training weights are changed.

The project data has 504 training images, 36 validation images and 36 held-out test images.
The source-photo audit checks that no photo crosses split boundaries.


In [ ]:
import tempfile

dataset_dir = project_dir / "project_milestone02/dataset"
data_audit = audit_dataset(dataset_dir)
display(pd.DataFrame([
    {"split": split, "images": values["images"], "labelled_faces": values["faces"]}
    for split, values in data_audit["splits"].items()
]))
print("Shared source photos:", data_audit["shared_source_photos"])

reference = json.loads((project_dir / "project_milestone03/results/metrics.json").read_text())
with tempfile.TemporaryDirectory() as temporary:
    resolved_checkpoint = resolve_checkpoint(checkpoint_path, training_zip_path, Path(temporary))
    checkpoint_digest = sha256(resolved_checkpoint)
print("Matches Aditi's reference checkpoint:", checkpoint_digest == reference["checkpoint_sha256"])
print("Matches the baseline dataset:", data_audit["dataset_sha256"] == reference["dataset"]["dataset_sha256"])

confidence_values = [0.10, 0.25, 0.50, 0.75]
nms_iou_values = [0.30, 0.50, 0.70]
agnostic_nms_values = [False, True]
device = "0" if torch.cuda.is_available() else "cpu"
batch_size = 8
image_size = 640
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output_dir = project_dir / "project_milestone03/results" / f"dario_sweep_{run_stamp}"
print("Device:", device)
print("Configurations:", len(confidence_values) * len(nms_iou_values) * len(agnostic_nms_values))
print("Output:", output_dir)


## 5. Run validation, lock the settings and evaluate test

Selection maximizes **validation micro-F1**, then recall, precision, mAP@0.5:0.95,
class-aware NMS and proximity to baseline. The selected settings are saved before test evaluation.

**Reading the scores:** mAP uses the common confidence floor of 0.001.
Precision, recall and F1 use the confidence specified in each row. mAP is repeated across
confidence rows sharing the same NMS settings. NMS IoU is separate from the fixed
IoU >= 0.50 used to match a prediction to a labelled face.

Only six validation passes are needed: predictions are reused across confidence cutoffs.
The script retains those low-confidence predictions for auditing.


In [ ]:
run_summary = run_experiment(
    checkpoint=checkpoint_path,
    training_zip=training_zip_path,
    dataset=dataset_dir,
    output=output_dir,
    device=device,
    batch=batch_size,
    image_size=image_size,
    confidences=confidence_values,
    nms_ious=nms_iou_values,
    agnostic_modes=agnostic_nms_values,
    gallery_count=10,
)
print("Selected using validation only:", run_summary["selection"]["config_id"])


## 6. Inspect the comparison and per-identity results

Precision and recall below are **fixed-threshold** scores. Mean matched IoU describes correct
same-identity detections only; missed faces and false positives are counted separately.


In [ ]:
validation_table = pd.read_csv(output_dir / "validation_sweep.csv")
score_columns = ["config_id", "map50", "map50_95", "precision", "recall", "f1", "mean_matched_iou", "fp", "fn"]
display(validation_table[score_columns].round(4))
display(Image(filename=str(output_dir / "validation_f1_heatmap.png"), width=1100))
print("Held-out test: predeclared baseline and selected configuration")
display(pd.read_csv(output_dir / "test_metrics.csv")[score_columns].round(4))
print("Per-identity held-out test scores")
per_class_table = pd.read_csv(output_dir / "per_class_metrics.csv", dtype={"identity": str})
display(per_class_table.loc[per_class_table["split"] == "test"].round(4))


## 7. Detection visualization gallery

Each image has a ground-truth panel and a prediction panel. Predicted boxes include identity IDs
and confidence scores. Green predictions match the correct identity at IoU >= 0.50;
red predictions are unmatched. Captions explain misses, wrong identities and overlapping predictions.

The gallery prioritizes actual failures and clear successes. Baseline failures can supplement
selected-configuration failures, with the source settings clearly marked. These images do not influence selection.


In [ ]:
gallery_rows = pd.read_csv(output_dir / "gallery/captions.csv")
for row in gallery_rows.to_dict("records"):
    display(Markdown(f"**{row['status']} — {row['image']}**  \nSettings: `{row['config_id']}` ({row['source']})."))
    display(Image(filename=str(output_dir / "gallery" / row["panel"]), width=1100))
    display(Markdown(row["caption"]))


## 8. Summary and teammate handoff

The generated Markdown summary has the sweep table, numerical interpretation, selected settings,
test comparison and gallery links. Yosephine can use it in the team's detection-results document.
Murat can reuse `selected_config.json` and the same checkpoint in the live demo.
Aditi's existing training log remains the source for training/validation loss curves; this sweep does not retrain.


In [ ]:
display(Markdown((output_dir / "hyperparameter_summary.md").read_text(encoding="utf-8")))
print("Checkpoint SHA-256:", run_summary["checkpoint_sha256"])
print("Dataset SHA-256:", run_summary["dataset"]["dataset_sha256"])
print("Recorded environment:", run_summary["environment"])


## 9. Package the results

The ZIP contains the comparison tables, exact configuration, generated interpretation,
ten annotated gallery examples and raw predictions. Preserve it with the checkpoint and
this executed notebook when integrating your section into the repository.


In [ ]:
import shutil

result_archive = Path(shutil.make_archive(str(output_dir), "zip", root_dir=output_dir.parent, base_dir=output_dir.name))
print("Result ZIP:", result_archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(result_archive))


## Limits and references

The validation and test scenes are synthetic collages built from a small set of source photographs.
Strong scores do not establish performance on real group photos. A correct-looking box can still have the wrong identity.
Class-agnostic NMS may remove duplicate identity predictions, but may also suppress a nearby real face.

- [Ultralytics validation metrics and settings](https://docs.ultralytics.com/modes/val/)
- [Ultralytics prediction and NMS settings](https://docs.ultralytics.com/modes/predict/)
- [Execution and verification status](verification.md)
